# Unit 4 Exercise 1: Build and Regularize a Small MLP

**Unit:** Unit 4: Neural Networks Fundamentals

**Why this exercise?** In the Unit 4 examples you watched someone else build,
overfit, and fix a neural network. Now it's your turn on a *real* dataset: the
Wisconsin breast-cancer dataset that ships with scikit-learn. You will build a
small Keras MLP, add one regularization technique (dropout), and compare the
two models — the exact workflow used in `07_early_stopping_regularization`.

## 📚 Learning Objectives

By completing this exercise, you will:
- Load and standardize a real sklearn dataset for a neural network
- Build and train a small Keras MLP classifier
- Add dropout regularization to an existing model
- Compare regularized vs unregularized models with honest numbers

## 🔗 Prerequisites

- ✅ Unit 4 examples, especially `07_early_stopping_regularization.ipynb`

## How to Work Through This Notebook

1. **Run every cell top-to-bottom first** — the notebook is fully working as
   shipped, so you see the complete workflow before touching anything.
2. Then complete the **"Your turn"** tasks at the bottom by *modifying and
   re-running* the cells (the helper function takes arguments for exactly the
   things you'll change).

---

## Part 0 — The Dataset

The breast-cancer dataset has **569 samples and 30 features** (measurements of
cell nuclei), with a binary label: malignant (0) or benign (1).

Two deliberate choices below:

- **We train on only 120 samples** and validate on the remaining 449. Small
  training sets are where overfitting shows up — with all 569 samples this
  exercise would be too easy to teach you anything about regularization.
- **We standardize the features** (zero mean, unit variance). The raw features
  have wildly different scales (some ~0.001, some ~2000), and neural networks
  train badly on unscaled inputs. Note the scaler is **fit on the training set
  only** — fitting it on validation data would leak information.

In [1]:
# Exercise setup: breast-cancer data with a DELIBERATELY small training set (120 rows)
# so that overfitting appears quickly and your regularization choices visibly matter.
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("=== Breast-cancer dataset ===")
data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_val, y_train, y_val = train_test_split(
    X, y, train_size=120, stratify=y, random_state=42
)

# Standardize using statistics from the training set only — fitting the scaler on all data would leak information.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)   # fit on train ONLY
X_val = scaler.transform(X_val)

print(f"Total samples : {X.shape[0]}  (features: {X.shape[1]})")
print(f"Training set  : {X_train.shape[0]} samples (deliberately small!)")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Class balance (train): {np.bincount(y_train)}  (0=malignant, 1=benign)")

=== Breast-cancer dataset ===
Total samples : 569  (features: 30)
Training set  : 120 samples (deliberately small!)
Validation set: 449 samples
Class balance (train): [45 75]  (0=malignant, 1=benign)


## Part 1 — Baseline MLP

A small MLP: 30 inputs → Dense(64, relu) → Dense(32, relu) → Dense(1, sigmoid).

The helper below takes `hidden`, `activation`, and `dropout` as arguments —
this is what you will reuse for every "Your turn" task, so read it carefully.
`keras.utils.set_random_seed(0)` inside the helper means every model starts
from the same initialization: any difference between two runs comes from what
*you* changed, not from random luck.

In [2]:
# Tooling for your experiments: build_model() lets you vary width, activation, and
# dropout; train_and_eval() records train/validation metrics for the summary table.
EPOCHS = 80
results = {}   # name -> metrics, filled by train_and_eval

def build_model(hidden=(64, 32), activation="relu", dropout=0.0):
    """Small MLP; change hidden sizes, activation, or dropout via arguments."""
    keras.utils.set_random_seed(0)
    layers = [Input(shape=(X_train.shape[1],))]
    for units in hidden:
        layers.append(Dense(units, activation=activation))
        if dropout > 0:
            layers.append(Dropout(dropout))
    layers.append(Dense(1, activation="sigmoid"))
    model = Sequential(layers)
    model.compile(optimizer="adam", loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

def train_and_eval(name, model, callbacks=None):
    history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                        epochs=EPOCHS, batch_size=16, verbose=0,
                        callbacks=callbacks or [])
    tr_loss, tr_acc = model.evaluate(X_train, y_train, verbose=0)
    va_loss, va_acc = model.evaluate(X_val, y_val, verbose=0)
    results[name] = {"train_acc": tr_acc, "val_acc": va_acc,
                     "gap": tr_acc - va_acc, "val_loss": va_loss,
                     "epochs_run": len(history.history["loss"])}
    print(f"{name}: train_acc={tr_acc:.3f}  val_acc={va_acc:.3f}  "
          f"gap={tr_acc - va_acc:+.3f}  val_loss={va_loss:.3f}")
    return history

print("=== Part 1: baseline MLP (no regularization) ===")
h_base = train_and_eval("Baseline (64,32) relu", build_model())
print("\nTrain accuracy is perfect but validation accuracy is lower —")
print("the model has started to memorize its 120 training rows.")

=== Part 1: baseline MLP (no regularization) ===


Baseline (64,32) relu: train_acc=1.000  val_acc=0.962  gap=+0.038  val_loss=0.142

Train accuracy is perfect but validation accuracy is lower —
the model has started to memorize its 120 training rows.


## Part 2 — Add One Regularization Technique

We add `Dropout(0.3)` after each hidden layer — the *only* change from the
baseline (same data, same seed, same 80 epochs). During training, 30% of each
hidden layer's activations are randomly zeroed, so the network can't lean on
any single unit to memorize a training row.

In [3]:
print("=== Part 2: same MLP + Dropout(0.3) ===")
h_drop = train_and_eval("Dropout 0.3", build_model(dropout=0.3))

=== Part 2: same MLP + Dropout(0.3) ===


Dropout 0.3: train_acc=1.000  val_acc=0.969  gap=+0.031  val_loss=0.130


## Part 3 — Compare

One table, two models. Look at three things:

1. **val_acc** — did regularization help where it counts?
2. **gap** (train minus val accuracy) — did memorization shrink?
3. **val_loss** — lower means better-calibrated predictions, even when
   accuracy barely moves.

In [4]:
# Compare your runs: the table shows accuracy and the train-minus-validation gap —
# a smaller gap with similar val accuracy means better generalization.
print("=== Part 3: with vs without dropout ===")
summary = pd.DataFrame(results).T
summary = summary[["epochs_run", "train_acc", "val_acc", "gap", "val_loss"]]
summary["epochs_run"] = summary["epochs_run"].astype(int)
print(summary.round(3).to_string())

base_acc = results["Baseline (64,32) relu"]["val_acc"]
drop_acc = results["Dropout 0.3"]["val_acc"]
print(f"\nDropout changed validation accuracy by {drop_acc - base_acc:+.3f}.")

=== Part 3: with vs without dropout ===
                       epochs_run  train_acc  val_acc    gap  val_loss
Baseline (64,32) relu          80        1.0    0.962  0.038     0.142
Dropout 0.3                    80        1.0    0.969  0.031     0.130

Dropout changed validation accuracy by +0.007.


### Reading the Result

On this run, dropout nudged validation accuracy from **0.962 to 0.969** and
lowered validation loss (0.142 → 0.130) — a small but real improvement, and
the train-vs-val gap shrank. Don't expect miracles: this dataset is fairly
easy even with 120 training samples, so regularization has limited room to
help. The *workflow* — baseline first, change one thing, compare a table — is
the point.

---

## 🎯 Your Turn

Do each task by **calling `build_model` with different arguments** in a new
cell (or editing a copy of the cells above), re-running, and comparing against
the table. Give each run a new name so `results` keeps all of them.

**Task 1 — Change the activation.** Train
`build_model(activation="tanh")` and compare with the relu baseline. Which
wins here, and by how much?

**Task 2 — Add a layer.** Train `build_model(hidden=(64, 32, 16))`. Does a
deeper network help or hurt with only 120 training samples? Check the gap
column, not just accuracy.

**Task 3 — Tune the dropout rate.** Try `dropout=0.1` and `dropout=0.5` and
compare with `0.3`. Is more dropout always better? Where does the trend stop?

**Bonus — Early stopping.** Import `EarlyStopping` from
`tensorflow.keras.callbacks` and pass
`callbacks=[EarlyStopping(monitor="val_loss", patience=10,
restore_best_weights=True)]` to `train_and_eval`. How many of the 80 epochs
did it actually need, and how does its val_loss compare with everything else?

When you're done (or stuck), compare with the instructor's worked solution.